In [1]:
!pip install ultralytics deepface tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 5.0 MB/s eta 0:00:00


In [2]:
import os, cv2, numpy as np, pandas as pd, torch
from tqdm import tqdm
from deepface import DeepFace
from PIL import Image
from ultralytics import YOLO

# Charge le modèle de pose (plus précis et rapide sur GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_pose = YOLO('yolov8n-pose.pt').to(device)

26-03-02 17:36:15 - Directory /root/.deepface has been created
26-03-02 17:36:15 - Directory /root/.deepface/weights has been created
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
def process_human_elements_yolo(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < 10:
            cap.release()
            return None

        indices = [int(total_frames * 0.2), int(total_frames * 0.5), int(total_frames * 0.8)]

        stats = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret or frame is None: continue

            # --- UTILISATION DE LA VERSION YOLO ICI ---
            # On appelle directement la logique YOLO-Pose pour éviter les erreurs d'import
            results = yolo_pose(frame, imgsz=320, verbose=False, conf=0.3)[0]

            face_count = 0
            max_face_area = 0
            has_body = 0

            if results.keypoints is not None:
                # results.keypoints.data contient les points [x, y, conf]
                for kpts in results.keypoints.data:
                    has_body = 1
                    # Points 0-4 = Nez, Yeux, Oreilles (Visage)
                    face_pts = kpts[:5]
                    if torch.any(face_pts[:, 2] > 0.5):
                        face_count += 1
                        x_min, y_min = torch.min(face_pts[:, :2], dim=0)[0]
                        x_max, y_max = torch.max(face_pts[:, :2], dim=0)[0]
                        area = (x_max - x_min) * (y_max - y_min) / (frame.shape[0] * frame.shape[1])
                        if area > max_face_area:
                            max_face_area = float(area)

            emotion = "none"
            if face_count > 0:
                try:
                    analysis = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False, silent=True)
                    emotion = analysis[0]['dominant_emotion']
                except: emotion = "unknown"

            stats.append((face_count, round(max_face_area, 4), emotion, has_body))

        cap.release()

        if not stats: return None

        # Agrégation
        face_counts = [s[0] for s in stats]
        face_areas = [s[1] for s in stats]
        emotions = [s[2] for s in stats]
        bodies = [s[3] for s in stats]
        dominant_emotion = max(set(emotions), key=emotions.count)

        return {
            'video_id': os.path.splitext(os.path.basename(video_path))[0],
            'avg_face_count': np.mean(face_counts),
            'max_face_coverage': np.max(face_areas),
            'dominant_emotion': dominant_emotion,
            'body_presence_score': np.mean(bodies)
        }
    except Exception as e:
        print(f"Erreur {video_path}: {e}")
        return None

In [4]:
from google.colab import drive
# 1. Montage du Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# --- BOUCLE PRINCIPALE ---
VIDEO_DIR = "/content/drive/MyDrive/hackathon/videos/"
paths = [os.path.join(r, f) for r, _, fs in os.walk(VIDEO_DIR) for f in fs if f.lower().endswith(('.mp4', '.mov'))]

# --- BOUCLE D'EXECUTION SECURISEE ---
results = []
for p in tqdm(paths): # Test sur 10
    data = process_human_elements_yolo(p)
    if data:
        results.append(data)

if len(results) > 0:
    df_human = pd.DataFrame(results)
    # On ne fait get_dummies que SI la colonne existe et contient des données
    if 'dominant_emotion' in df_human.columns:
        df_human = pd.get_dummies(df_human, columns=['dominant_emotion'])

    df_human.to_csv("/content/drive/MyDrive/hackathon/features_humain.csv", index=False)
    print(f"✅ Terminé : {len(df_human)} vidéos traitées.")
else:
    print("❌ Aucune donnée n'a été extraite. Vérifie les chemins de tes vidéos.")

  0%|          | 2/1686 [00:07<1:29:04,  3.17s/it]Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5


26-03-02 17:37:06 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...



100%|██████████| 5.98M/5.98M [00:00<00:00, 76.5MB/s]
100%|██████████| 1686/1686 [1:21:50<00:00,  2.91s/it]


✅ Terminé : 1686 vidéos traitées.
